In [1]:
import sys
sys.path.append("..")

In [3]:
from itertools import combinations

import pandas as pd
from sklearn.metrics import average_precision_score

from src.labels import EXTRA_FEATURE_COLUMNS, FEATURE_COLUMNS
from src.train import fit_model

samples = pd.read_parquet("../data/samples.parquet")
samples["current_lightning"] = samples["flash_now"].astype("float32")
feature_columns = FEATURE_COLUMNS + ["current_lightning"] + EXTRA_FEATURE_COLUMNS
all_days = sorted(samples["day"].unique())

curve_rows = []
for test_day in all_days:
    other_days = [day for day in all_days if day != test_day]
    test_samples = samples[samples["day"] == test_day]
    for train_day_count in range(1, len(other_days) + 1):
        for train_days in combinations(other_days, train_day_count):
            train_samples = samples[samples["day"].isin(train_days)]
            model = fit_model(train_samples, feature_columns)
            probabilities = model.predict_proba(test_samples[feature_columns])[:, 1]
            curve_rows.append({
                "test_day": test_day,
                "train_day_count": train_day_count,
                "pr_auc": average_precision_score(test_samples["label"], probabilities),
            })

learning_curve = pd.DataFrame(curve_rows)
print(learning_curve.groupby(["test_day", "train_day_count"])["pr_auc"].mean().unstack().round(3))
print(learning_curve.groupby("train_day_count")["pr_auc"].mean().round(3))

train_day_count      1      2      3      4
test_day                                   
2026-07-18       0.424  0.474  0.493  0.498
2026-07-19       0.382  0.429  0.437  0.445
2026-07-20       0.401  0.451  0.470  0.473
2026-07-21       0.349  0.378  0.385  0.384
2026-07-22       0.091  0.103  0.110  0.106
train_day_count
1    0.329
2    0.367
3    0.379
4    0.381
Name: pr_auc, dtype: float64
